# 🛰️ 01 — Download Sentinel-2 Data
### Amazon Deforestation Change Detection Pipeline
This notebook downloads two Sentinel-2 scenes of **Rondônia, Brazil** from 2019 and 2024
via the free Copernicus Data Space API.

> **No GPU required.** All steps run on CPU.

---
**Pipeline overview:**
1. ⬇️ `01_download_data.ipynb` ← *you are here*
2. 🔧 `02_prepare_scenes.ipynb`
3. 🌳 `03_detect_changes.ipynb`
4. 🎨 `04_visualise.ipynb`


## ⚙️ Configuration
Edit the bounding box or date ranges here if you want a different location or time period.

In [1]:
from pathlib import Path

# ── Area of Interest ──────────────────────────────────────────────────────────
# Rondônia, Brazil — the "fishbone" deforestation corridor
# Format: (lon_min, lat_min, lon_max, lat_max)
BBOX = (-63.5, -11.5, -62.5, -10.5)

# ── Date ranges — dry season (June–September) for cloud-free imagery ──────────
DATE_RANGES = {
    "2019": ("2019-07-01", "2019-09-30"),   # Pre-deforestation baseline
    "2024": ("2024-07-01", "2024-09-30"),   # Post-deforestation
}

# ── Sentinel-2 bands ──────────────────────────────────────────────────────────
# B04=Red  B03=Green  B02=Blue  B08=NIR (Near-Infrared, used for NDVI)
BANDS = ["B04", "B03", "B02", "B08"]

MAX_CLOUD_COVER = 10   # percent — only scenes with < 10% cloud cover

OUTPUT_DIR = Path("data/raw")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("✅ Configuration set")


✅ Configuration set


## 📡 Step 1 — Search for Available Scenes
Queries the Copernicus catalogue for cloud-free Sentinel-2 scenes over the AOI.

In [2]:
import requests

OPENSEARCH_URL = "https://catalogue.dataspace.copernicus.eu/resto/api/collections/Sentinel2/search.json"

def search_scenes(year, date_start, date_end):
    lon_min, lat_min, lon_max, lat_max = BBOX
    params = {
        "startDate":       date_start,
        "completionDate":  date_end,
        "processingLevel": "S2MSI2A",
        "cloudCover":      f"[0,{MAX_CLOUD_COVER}]",
        "box":             f"{lon_min},{lat_min},{lon_max},{lat_max}",
        "maxRecords":      5,
        "sortParam":       "cloudCover",
        "sortOrder":       "ascending",
    }
    print(f"\n🔍 Searching for {year} scenes...")
    resp = requests.get(OPENSEARCH_URL, params=params, timeout=30)
    resp.raise_for_status()
    features = resp.json().get("features", [])
    print(f"   Found {len(features)} scene(s) with < {MAX_CLOUD_COVER}% cloud cover")
    for f in features:
        p = f["properties"]
        print(f"   → {p.get('title','?')[:60]}  cloud={p.get('cloudCover','?')}%  date={p.get('startDate','?')[:10]}")
    return features

scenes = {}
for year, (start, end) in DATE_RANGES.items():
    scenes[year] = search_scenes(year, start, end)



🔍 Searching for 2019 scenes...


HTTPError: 403 Client Error: Forbidden for url: https://catalogue.dataspace.copernicus.eu/resto/api/collections/Sentinel2/search.json?startDate=2019-07-01&completionDate=2019-09-30&processingLevel=S2MSI2A&cloudCover=%5B0%2C10%5D&box=-63.5%2C-11.5%2C-62.5%2C-10.5&maxRecords=5&sortParam=cloudCover&sortOrder=ascending

## 📥 Step 2 — Download Scenes
Downloads the lowest cloud-cover scene for each year and extracts the RGB+NIR bands.

In [ ]:
import zipfile

DOWNLOAD_BASE = "https://zipper.dataspace.copernicus.eu/zip"

def download_scene(scene, year):
    out_dir = OUTPUT_DIR / year
    out_dir.mkdir(parents=True, exist_ok=True)

    product_id  = scene["id"]
    cloud_cover = scene["properties"].get("cloudCover", "?")
    date        = scene["properties"].get("startDate", "?")[:10]

    print(f"\n📥 Downloading {year} scene (cloud={cloud_cover}%, date={date})")
    zip_url  = f"{DOWNLOAD_BASE}?productId={product_id}"
    zip_path = out_dir / f"{year}_scene.zip"

    with requests.get(zip_url, stream=True, timeout=120) as r:
        r.raise_for_status()
        total      = int(r.headers.get("content-length", 0))
        downloaded = 0
        with open(zip_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                f.write(chunk)
                downloaded += len(chunk)
                if total:
                    print(f"\r   Progress: {downloaded/total*100:.1f}%", end="", flush=True)
    print()

    with zipfile.ZipFile(zip_path, "r") as zf:
        for member in zf.namelist():
            if any(f"_{b}_" in member or member.endswith(f"_{b}.jp2") for b in BANDS):
                zf.extract(member, out_dir)
                print(f"   Extracted: {Path(member).name}")

    zip_path.unlink()
    print(f"   ✅ {year} scene ready in {out_dir}")

try:
    for year, year_scenes in scenes.items():
        if year_scenes:
            download_scene(year_scenes[0], year)
        else:
            print(f"⚠️  No scenes found for {year}")
except requests.HTTPError as e:
    if e.response.status_code in (401, 403):
        print("\n⚠️  Authentication required. See manual instructions below.")
    else:
        raise


## 🤲 Manual Download (if API authentication fails)

If the download above fails, follow these steps:

1. Go to **[browser.dataspace.copernicus.eu](https://browser.dataspace.copernicus.eu)**
2. Create a **free ESA account** (takes ~2 minutes)
3. Search: `Rondônia, Brazil` · Data: `Sentinel-2 L2A` · Cloud cover: `0–10%`
4. Date range: **2019-07-01 to 2019-09-30** → download best result → extract to `data/raw/2019/`
5. Date range: **2024-07-01 to 2024-09-30** → download best result → extract to `data/raw/2024/`

**Alternative:** Run the cell below to generate synthetic demo data so you can test the full pipeline immediately.


In [5]:
# ── OPTIONAL: Generate synthetic demo data for testing ────────────────────────
# Run this cell if you don't have real Sentinel-2 data yet.
# The pipeline will work end-to-end with synthetic data.

import numpy as np
from rasterio.transform import from_bounds
import rasterio

def create_synthetic_bands(year):
    np.random.seed(42 if year == "2019" else 99)
    H, W = 512, 512
    out_dir = Path("data/raw") / year
    out_dir.mkdir(parents=True, exist_ok=True)
    transform = from_bounds(-63.5, -11.5, -62.5, -10.5, W, H)
    profile = {"driver": "GTiff", "dtype": "uint16", "width": W, "height": H,
               "count": 1, "crs": "EPSG:4326", "transform": transform}

    if year == "2019":
        vals = {"B04": (600, 50), "B03": (900, 80), "B02": (500, 40), "B08": (3500, 200)}
    else:
        vals = {"B04": (1200, 150), "B03": (1100, 120), "B02": (800, 100), "B08": (2200, 300)}

    for band, (mean, std) in vals.items():
        data = np.random.normal(mean, std, (H, W)).clip(0, 10000).astype(np.uint16)
        if year == "2024":
            for i in range(0, W, 40):
                data[:, i:i+8] = 3000  # cleared strips
        with rasterio.open(out_dir / f"{band}.tif", "w", **profile) as dst:
            dst.write(data[np.newaxis, :])
    print(f"✅ Synthetic bands created for {year} in {out_dir}")

create_synthetic_bands("2019")
create_synthetic_bands("2024")
print("\n✅ Done. Proceed to: 02_prepare_scenes.ipynb")


✅ Synthetic bands created for 2019 in data\raw\2019
✅ Synthetic bands created for 2024 in data\raw\2024

✅ Done. Proceed to: 02_prepare_scenes.ipynb
